In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load and prepare the data
df = pd.read_csv("top_5000_mi_based_subset.csv") 
X = df.drop(columns=["readmitted"]).values.astype(float)
y = df["readmitted"].values.astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 2. Convert to PyTorch tensors and create DataLoaders
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds  = TensorDataset(X_test_tensor,  y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

# 3. Define the neural network
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)




# 4. Instantiate model, loss, optimizer
device = torch.device("cuda")  # or "cuda" if GPU is available
model = SimpleNN(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 5. Training loop
num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch:02d}/{num_epochs}, Loss: {epoch_loss:.4f}")

# 6. Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        predicted = (preds > 0.5).float()
        total += yb.size(0)
        correct += (predicted == yb).sum().item()
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


Epoch 01/20, Loss: 0.6188
Epoch 02/20, Loss: 0.5673
Epoch 03/20, Loss: 0.5544
Epoch 04/20, Loss: 0.5482
Epoch 05/20, Loss: 0.5431
Epoch 06/20, Loss: 0.5433
Epoch 07/20, Loss: 0.5370
Epoch 08/20, Loss: 0.5352
Epoch 09/20, Loss: 0.5337
Epoch 10/20, Loss: 0.5322
Epoch 11/20, Loss: 0.5317
Epoch 12/20, Loss: 0.5346
Epoch 13/20, Loss: 0.5252
Epoch 14/20, Loss: 0.5278
Epoch 15/20, Loss: 0.5217
Epoch 16/20, Loss: 0.5245
Epoch 17/20, Loss: 0.5196
Epoch 18/20, Loss: 0.5209
Epoch 19/20, Loss: 0.5186
Epoch 20/20, Loss: 0.5178
Test Accuracy: 0.7310


In [73]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("top_5000_mi_based_subset.csv")  # replace with your file path
X = df.drop(columns=["readmitted"]).values.astype(float)
y = df["readmitted"].values.astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 2. Convert to PyTorch tensors and create DataLoaders (Ensure this part is run)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds  = TensorDataset(X_test_tensor,  y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

# -----------------------------------------
# 3. Define the NEW ResNet-like Network
# -----------------------------------------

class ResidualBlock(nn.Module):
    """A basic residual block for tabular data."""
    def __init__(self, input_dim, output_dim, dropout_rate=0.2):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, output_dim)
        self.bn1 = nn.BatchNorm1d(output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear2 = nn.Linear(output_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

        # Shortcut connection: Adjust dimensions if input_dim != output_dim
        if input_dim != output_dim:
            self.shortcut = nn.Sequential(
                nn.Linear(input_dim, output_dim),
                nn.BatchNorm1d(output_dim)
            )
        else:
            self.shortcut = nn.Identity() # No adjustment needed

    def forward(self, x):
        identity = x # Store the input for the shortcut

        out = self.linear1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.linear2(out)
        out = self.bn2(out)

        # Add the shortcut connection
        shortcut_out = self.shortcut(identity)
        out += shortcut_out
        out = self.relu(out) # Final activation for the block
        return out

class ResNetTabular(nn.Module):
    """A ResNet-inspired model for tabular data."""
    def __init__(self, input_dim, hidden_dims=[128, 128, 64], dropout_rate=0.2):
        super().__init__()
        self.initial_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.ReLU()
        )

        layers = []
        current_dim = hidden_dims[0]
        for h_dim in hidden_dims[1:]: # Start from the second hidden dim
            layers.append(ResidualBlock(current_dim, h_dim, dropout_rate))
            current_dim = h_dim # Update current_dim for the next block

        self.residual_blocks = nn.Sequential(*layers)
        self.final_layer = nn.Linear(current_dim, 1) # Output logits

    def forward(self, x):
        x = self.initial_layer(x)
        x = self.residual_blocks(x)
        x = self.final_layer(x)
        return x # Output logits (use BCEWithLogitsLoss)

# -----------------------------------------
# 4. Instantiate model, loss, optimizer
# -----------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Correct check

# Define hidden dimensions for the blocks
# You can experiment with depth and width here
hidden_dims_config = [128, 128, 64, 64] # e.g., Initial layer -> 128, ResBlock(128->128), ResBlock(128->64), ResBlock(64->64)

model = ResNetTabular(
    input_dim=X_train.shape[1],
    hidden_dims=hidden_dims_config,
    dropout_rate=0.25 # Adjust dropout if needed
).to(device)

# Use BCEWithLogitsLoss (recommended with ResNet/logits output)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3) # Adam is usually a good start

print("Model Architecture:")
print(model)
print(f"\nUsing device: {device}")

# -----------------------------------------
# 5. Training loop (Identical structure to before)
# -----------------------------------------
num_epochs = 50 # Adjust as needed
for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb) # Model outputs logits
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch:02d}/{num_epochs}, Loss: {epoch_loss:.4f}")

# -----------------------------------------
# 6. Evaluation (Adjusted for logits output)
# -----------------------------------------
model.eval()
correct = 0
total = 0
all_preds = []
all_targets = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb) # Model outputs logits
        preds = torch.sigmoid(logits) # Apply sigmoid to get probabilities
        predicted = (preds > 0.5).float() # Threshold probabilities

        total += yb.size(0)
        correct += (predicted == yb).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

accuracy = correct / total
print(f"\nTest Accuracy: {accuracy:.4f}")

# Optional: Further evaluation metrics (if needed)
# from sklearn.metrics import classification_report
# print(classification_report(all_targets, all_preds))

Model Architecture:
ResNetTabular(
  (initial_layer): Sequential(
    (0): Linear(in_features=20, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (residual_blocks): Sequential(
    (0): ResidualBlock(
      (linear1): Linear(in_features=128, out_features=128, bias=True)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (dropout): Dropout(p=0.25, inplace=False)
      (linear2): Linear(in_features=128, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Identity()
    )
    (1): ResidualBlock(
      (linear1): Linear(in_features=128, out_features=64, bias=True)
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (dropout): Dropout(p=0.25, inplace=False)
      (linear2): Li

In [74]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Load
df = pd.read_csv('top_5000_mi_based_subset.csv')

# 2. Define target and features
target = 'readmitted'
features = [c for c in df.columns if c != target]

# 3. Correct categorical feature names
cat_features = [
    'race',
    'gender',
    'age',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'diag_1',
    'max_glu_serum',
    'A1Cresult',
    'insulin',
    'change',
    'diabetesMed'
]

# 4. Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target],
    test_size=0.2,
    random_state=42,
    stratify=df[target]
)

# 5. Initialize CatBoost
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.001,
    depth=9,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

# 6. Fit with early stopping
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    use_best_model=True
)

# 7. Evaluate
y_pred = model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 8. Show importances
print("\nFeature Importances:")
print(model.get_feature_importance(prettified=True))


0:	learn: 0.7387500	test: 0.7300000	best: 0.7300000 (0)	total: 28.5ms	remaining: 28.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.745
bestIteration = 8

Shrink model to first 9 iterations.
Test Accuracy: 0.745

Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.60      0.63       360
           1       0.79      0.82      0.81       640

    accuracy                           0.74      1000
   macro avg       0.72      0.71      0.72      1000
weighted avg       0.74      0.74      0.74      1000


Feature Importances:
                  Feature Id  Importances
0           number_inpatient    47.071453
1   discharge_disposition_id    22.691202
2                        age     7.523721
3              max_glu_serum     4.589677
4                       race     3.793799
5        admission_source_id     3.032761
6                  A1Cresult     2.449149
7           number_emergency     2.395578
8          ad

In [77]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import sys # For version checking

# 1. Install (once):
#    pip install tabpfn

# 2. Import the classifier
try:
    from tabpfn import TabPFNClassifier
    import tabpfn # Import the base library to check version
except ImportError:
    print("TabPFN library not found. Please install it: pip install tabpfn")
    exit()

# Optional: Print installed version
try:
    print(f"Using tabpfn version: {tabpfn.__version__}")
except AttributeError:
     print("Could not determine tabpfn version automatically.")


# 3. Load your balanced data
try:
    df = pd.read_csv('top_5000_mi_based_subset.csv')
    X = df.drop(columns=['readmitted']).values
    y = df['readmitted'].values
except FileNotFoundError:
    print("Error: balanced_sample.csv not found. Please ensure the file is in the correct directory.")
    exit()
except KeyError:
    print("Error: 'readmitted' column not found in balanced_sample.csv.")
    exit()


# 4. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Initialize (SIMPLIFIED)
#    Remove arguments not accepted by your installed version's __init__
print("\nInitializing TabPFNClassifier (simplified)...")
try:
    clf = TabPFNClassifier(
        device='cuda'  # Keep 'cuda' or change to 'cpu' if needed
        # REMOVED: N_ensemble_configurations, seed, fit_at_predict_time
    )
    print("TabPFNClassifier initialized successfully.")
except TypeError as e:
    print(f"\nFATAL ERROR during initialization: {e}")
    print("Even the simplified initialization failed.")
    print("Please verify your 'tabpfn' installation and consult the documentation specific")
    print("to your installed version for the correct way to instantiate TabPFNClassifier.")
    exit()
except Exception as e:
    print(f"\nAn unexpected error occurred during initialization: {e}")
    exit()


# 6. Fit & predict
print("\nFitting TabPFNClassifier...")
try:
    clf.fit(X_train, y_train)
    print("Fit complete.")
    print("\nPredicting with TabPFNClassifier...")
    y_pred = clf.predict(X_test)
    print("Prediction complete.")
except Exception as e:
    print(f"\nError during fit or predict: {e}")
    exit()


# 7. Evaluate
print("\nEvaluating...")
try:
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    print(f"\nAccuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(report)
except Exception as e:
    print(f"\nError during evaluation: {e}")

Using tabpfn version: 2.0.8

Initializing TabPFNClassifier (simplified)...
TabPFNClassifier initialized successfully.

Fitting TabPFNClassifier...
Fit complete.

Predicting with TabPFNClassifier...
Prediction complete.

Evaluating...

Accuracy: 0.7420

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.59      0.62       360
           1       0.78      0.82      0.80       640

    accuracy                           0.74      1000
   macro avg       0.72      0.71      0.71      1000
weighted avg       0.74      0.74      0.74      1000



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
# Choose one: LightGBM is often faster, XGBoost is also excellent
import lightgbm as lgb
# import xgboost as xgb

# 1. Load your balanced data
df = pd.read_csv('top_5000_mi_based_subset.csv')
X = df.drop(columns=['readmitted']).values
y = df['readmitted'].values

# 2. Apply Scaling (Recommended for consistency, though GBMs less sensitive)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Split (using scaled data)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Initialize LightGBM Classifier (Example)
print("\nTraining LightGBM...")
# Basic parameters, can be tuned later
lgb_clf = lgb.LGBMClassifier(random_state=42)


# 5. Fit
lgb_clf.fit(X_train, y_train)
# xgb_clf.fit(X_train, y_train)

# 6. Predict
y_pred_lgb = lgb_clf.predict(X_test)
# y_pred_xgb = xgb_clf.predict(X_test)

# 7. Evaluate
print("\nLightGBM Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print(classification_report(y_test, y_pred_lgb))

# print("\nXGBoost Results:")
# print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
# print(classification_report(y_test, y_pred_xgb))

# --- Basic Hyperparameter Tuning (Example for LightGBM) ---
# If default accuracy isn't 0.73, try tuning key parameters:
# lgb_clf_tuned = lgb.LGBMClassifier(
#     random_state=42,
#     n_estimators=200,       # More trees
#     learning_rate=0.05,     # Lower learning rate
#     num_leaves=31,          # Default, can increase/decrease
#     max_depth=-1,           # Default (no limit), can limit
#     reg_alpha=0.1,          # L1 regularization
#     reg_lambda=0.1          # L2 regularization
# )
# You would then fit and predict with lgb_clf_tuned
# Consider using GridSearchCV or RandomizedSearchCV for systematic tuning.


Training LightGBM...
[LightGBM] [Info] Number of positive: 2000, number of negative: 2000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000465 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 545
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

LightGBM Results:
Accuracy: 0.649
              precision    recall  f1-score   support

           0       0.66      0.61      0.64       500
           1       0.64      0.69      0.66       500

    accuracy                           0.65      1000
   macro avg       0.65      0.65      0.65      1000
weighted avg       0.65      0.65      0.65      1000



/home/seyam-omar/jupyenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [78]:
import optuna, pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier

# ---------------- data ----------------
df      = pd.read_csv("top_5000_mi_based_subset.csv")
target  = "readmitted"
X, y    = df.drop(columns=[target]).values, df[target].values

cat_cols = ['race','gender','age','admission_type_id','discharge_disposition_id',
            'admission_source_id','diag_1','max_glu_serum','A1Cresult',
            'insulin','change','diabetesMed']
cat_idx  = [df.drop(columns=[target]).columns.get_loc(c) for c in cat_cols]

kf = StratifiedKFold(5, shuffle=True, random_state=42)

# ------------- objective -------------
def objective(trial):
    params = {
        "iterations":       trial.suggest_int("iters", 800, 2500),
        "learning_rate":    trial.suggest_float("lr", 0.02, 0.3, log=True),
        "depth":            trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":      trial.suggest_int("l2", 1, 15),
        "subsample":        trial.suggest_float("sub", 0.6, 1.0),
        "colsample_bylevel":trial.suggest_float("col", 0.5, 1.0),
        "loss_function":    "Logloss",
        "eval_metric":      "Accuracy",
        "random_seed":      42,
        "verbose":          False
    }

    fold_acc = []
    for tr_idx, val_idx in kf.split(X, y):
        model = CatBoostClassifier(**params)
        model.fit(X[tr_idx], y[tr_idx],
                  cat_features=cat_idx,
                  verbose=False)
        preds = model.predict(X[val_idx]).astype(int)
        fold_acc.append((preds == y[val_idx]).mean())

    mean_acc = np.mean(fold_acc)

    # ---------- verbose print ----------
    print(f"Trial {trial.number:02d}  |  acc={mean_acc:.4f}  |  "
          f"iters={params['iterations']}, depth={params['depth']}, "
          f"lr={params['learning_rate']:.3f}, l2={params['l2_leaf_reg']}, "
          f"sub={params['subsample']:.2f}, col={params['colsample_bylevel']:.2f}")

    return mean_acc

# ------------- run search -------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\nBest 5‑fold CV accuracy:", study.best_value)
print("Best hyper‑params:\n", study.best_params)


[I 2025-04-21 10:23:44,015] A new study created in memory with name: no-name-3ce9a318-dac0-42d2-9000-800622218694


  0%|          | 0/30 [00:00<?, ?it/s]

Trial 00  |  acc=0.7466  |  iters=1387, depth=8, lr=0.041, l2=7, sub=0.79, col=0.75
[I 2025-04-21 10:25:14,028] Trial 0 finished with value: 0.7466000000000002 and parameters: {'iters': 1387, 'lr': 0.041288129795629756, 'depth': 8, 'l2': 7, 'sub': 0.7877661842271328, 'col': 0.7530095918577441}. Best is trial 0 with value: 0.7466000000000002.
Trial 01  |  acc=0.7478  |  iters=1881, depth=6, lr=0.028, l2=8, sub=0.72, col=0.71
[I 2025-04-21 10:26:30,078] Trial 1 finished with value: 0.7478 and parameters: {'iters': 1881, 'lr': 0.028420037093887428, 'depth': 6, 'l2': 8, 'sub': 0.7228703597390267, 'col': 0.7113775984301395}. Best is trial 1 with value: 0.7478.
Trial 02  |  acc=0.7360  |  iters=1539, depth=8, lr=0.173, l2=3, sub=0.93, col=0.81
[I 2025-04-21 10:28:12,895] Trial 2 finished with value: 0.736 and parameters: {'iters': 1539, 'lr': 0.1734952529081107, 'depth': 8, 'l2': 3, 'sub': 0.9328091213756117, 'col': 0.8094864285713164}. Best is trial 1 with value: 0.7478.
Trial 03  |  acc=0.

KeyboardInterrupt: 

In [62]:
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
import numpy as np

# Load dataset
df = pd.read_csv("filtered_data.csv")
X = df.drop(columns=["readmitted"]).values
y = df["readmitted"].values.reshape(-1, 1)

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 90:10 split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.1, random_state=42)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=512, shuffle=True)


class DeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 512)
        self.layer2 = nn.Linear(512, 128)
        self.layer3 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # Optional if you use BCE, not BCEWithLogits

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        self.embeddings = x.clone().detach()  # 64-dimensional
        return self.sigmoid(self.output(x))


model = DeepMLP(input_dim=X.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training with verbose logging
model.train()
for epoch in range(1, 21):  # Epochs 1 to 20
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    avg_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch:02d}/20] - Loss: {avg_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(torch.tensor(X_scaled, dtype=torch.float32))
    pred_labels = (preds >= 0.5).float()
    accuracy = accuracy_score(torch.tensor(y).numpy(), pred_labels.numpy())
    print("Test Accuracy on full data:", round(accuracy, 4))

    # Get full-dataset embeddings and losses
    embeddings_np = model.embeddings.numpy()
    losses = nn.BCELoss(reduction='none')(preds, torch.tensor(y, dtype=torch.float32)).numpy().flatten()

# KMeans clustering on embeddings
k = 2500
kmeans = KMeans(n_clusters=k, random_state=42)
cluster_ids = kmeans.fit_predict(embeddings_np)

# Select samples per cluster
selected_indices = set()
for i in range(k):
    cluster_mask = (cluster_ids == i)
    cluster_indices = np.where(cluster_mask)[0]
    if len(cluster_indices) == 0:
        continue
    cluster_emb = embeddings_np[cluster_mask]
    cluster_losses = losses[cluster_mask]
    centroid = kmeans.cluster_centers_[i]

    # Closest to centroid
    closest_idx = cluster_indices[np.argmin(np.linalg.norm(cluster_emb - centroid, axis=1))]

    # Highest loss in cluster
    highest_loss_idx = cluster_indices[np.argmax(cluster_losses)]

    selected_indices.add(closest_idx)
    selected_indices.add(highest_loss_idx)

# Save selected 5000 samples
selected_df = df.iloc[list(selected_indices)]
selected_df.to_csv("best_5000_samples.csv", index=False)
print("Saved best 5000 samples to 'best_5000_samples.csv'")


Epoch [01/20] - Loss: 0.4400
Epoch [02/20] - Loss: 0.4172
Epoch [03/20] - Loss: 0.4154
Epoch [04/20] - Loss: 0.4135
Epoch [05/20] - Loss: 0.4124
Epoch [06/20] - Loss: 0.4112
Epoch [07/20] - Loss: 0.4090
Epoch [08/20] - Loss: 0.4082
Epoch [09/20] - Loss: 0.4056
Epoch [10/20] - Loss: 0.4054
Epoch [11/20] - Loss: 0.4026
Epoch [12/20] - Loss: 0.4002
Epoch [13/20] - Loss: 0.3980
Epoch [14/20] - Loss: 0.3952
Epoch [15/20] - Loss: 0.3931
Epoch [16/20] - Loss: 0.3912
Epoch [17/20] - Loss: 0.3878
Epoch [18/20] - Loss: 0.3860
Epoch [19/20] - Loss: 0.3835
Epoch [20/20] - Loss: 0.3809
Test Accuracy on full data: 0.8464
Saved best 5000 samples to 'best_5000_samples.csv'


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("top_5000_mi_based_subset.csv")

# Split features and target
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

# Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest with tuned parameters
rf_clf = RandomForestClassifier(
    n_estimators=10000,
    max_depth=100,
    min_samples_split=2,
    max_features='sqrt',
    random_state=42
)
rf_clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Tuned Random Forest Test Accuracy: {accuracy:.4f}")


Tuned Random Forest Test Accuracy: 0.7480
